## Data_preparation

이 노트북에서는 Supervised Fine-Tuning(SFT) 및 Direct Preference Optimization(DPO) 모델을 학습하기 위해 필요한 데이터를 준비합니다.  

구체적으로 다음과 같은 내용을 진행합니다:
 - 원본 데이터셋 로드 및 전처리
 - Instruction 포맷팅 (QA, 요약)
 - Chosen / Rejected 선호 데이터셋 구성 (DPO)

### 1. 라이브러리 import

In [5]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 18.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
# 데이터 전처리를 위한 필수 라이브러리 임포트
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset

### 2. 데이터 폴더 구조 설정

In [7]:
# 데이터 경로 설정
RAW_DATA_PATH = 'data/raw'
PROCESSED_DATA_PATH = 'data/processed'

os.makedirs(RAW_DATA_PATH, exist_ok=True)
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

### 3. 원본 데이터셋 로드 및 기본 전처리

In [8]:
# QA 데이터 로드
df_qa = pd.read_csv(os.path.join(RAW_DATA_PATH, 'KorQuAd.csv'), encoding='utf-8', engine='python', on_bad_lines='skip')
df_qa.head()

,context,question,answer
0,1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로...,1. 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?\n2. 바그너는 교향곡...,1. 교향곡\n2. 1악장\n3. 베토벤의 교향곡 9번\n4. 파우스트\n5. 합창...
1,서일본 여객철도 어번 네트워크의 한 축을 구성하고 있다. 오사카 시가지 남부의 철도...,1. 한와 선은 오사카 덴노지에서 어디까지 가는 노선인가?\n2. 심한 구배 구간이...,1. 와카야마\n2. 오노야마 고개\n3. 와카야마 역\n4. 1994년\n5. 와...
2,"1939년 9월 1일 현지 시간 4시 48분, 전함 슐레스비히-홀슈타인이 폴란드 주...","1. 1939년 9월 1일, 전함 슐레스비히-홀슈타인의 일제 포격으로 시작된 폴란드...",1. 제2차 세계 대전\n2. 보이시에흐 나이사레크\n3. 빌헬름 헤닝센\n4. 1...
3,"헨리 구스타프 몰래슨(영어: Henry Gustav Molaison, 1926년 2...",1. H.M으로 알려진 환자의 뇌가 관리되고 있는 지역은?\n2. 미국의 헨리 구스...,1. 미국 샌디에이고\n2. 해마를 포함한 내측측두엽\n3. 뇌전증\n4. 1957...
4,"초우라늄 원소의 존재는 1934년, 엔리코 페르미가 그의 연구를 바탕으로 제안한 것...",1. 초우라늄 원소가 존재할 가능성을 처음 제안한 사람은?\n2. 산화 6의 산화 ...,1. 엔리코 페르미\n2. 백금\n3. 탄탈럼\n4. 1934년\n5. 1950년대 후반


In [4]:
# 요약 데이터 로드
df_summary = pd.read_csv(os.path.join(RAW_DATA_PATH, 'Summarization.csv'), encoding='utf-8', on_bad_lines='skip')
df_summary.head()

,context,summary
0,원고가 소속회사의 노동조합에서 분규가 발생하자 노조활동을 구실로 정상적인 근무를 해...,원고가 주동하여 회사업무능률을 저해하고 회사업무상의 지휘명령에 위반하였다면 이에 ...
1,수출입업체인 원고가 의류제품을 제조ㆍ수출함에 있어 같은 그룹내 종합무역상사인 소외 ...,수출입업체인 원고가 의류제품을 제조ㆍ수출함에 있어 소외 회사의 직수출실적을 지원하기...
2,가등기담보권자가 제소전 화해조항에 따라 자기 명의로 소유권이전의 본등기를 경료한 후...,가등기담보권자가 제소전 화해조항에 의해 자기 명의로 소유권이전의 본등기를 경료하고 ...
3,가. 부가가치세법 제22조 제3항 단서에 제1호와 제2호가 동시에 해당한다는 뜻은 ...,부가가치세법 제22조 제3항 단서에 제1호와 제2호가 동시에 해당한다는 의미는 제1...
4,소득세법 제116조 제1항의 규정에 의하면 정부는 과세표준확정신고를 하여야 할 자에...,소득세법 제116조 제1항에 따르면 정부는 과세표준확정신고를 해야 할 자에 대해 당...


### 4. **Instruction Prompt** 형식에 맞게 데이터셋 포맷팅
LLaMA Instruct 모델은 특정 형식의 프롬프트 구조를 따릅니다.  
이 포맷은 `system`, `user`, `assistant` 세 가지 역할(role)을 명시적으로 나누고,  
각 역할은 **특수 토큰**으로 감싸진 메타 정보로 구분됩니다.


전체 구조

```text
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{assistant_response}<|eot_id|>
```

`<|begin_of_text|>:` 전체 시퀀스의 시작을 나타냅니다.  
`<|start_header_id|>role<|end_header_id|>`: system, user, assistant 역할 정보를 명시합니다.  
`<|eot_id|>`: 해당 역할의 발화를 종료하는 토큰입니다.  

`system`: 시스템 메시지. 모델에게 행동 방식을 지시합니다. 예: "당신은 유용한 AI 어시스턴트입니다."  
`user`: 사용자 메시지. 실제 질문이나 요청이 이 영역에 들어갑니다.  
`assistant`: 모델이 응답해야 할 공간. 이 영역에 대해 학습이 이루어지거나, 추론 시 출력이 생성됩니다.  

#### QA 데이터셋 포맷팅

In [9]:
# 문장의 최대 길이 설정
MAX_LENGTH = 1024

In [10]:
# Llama instruct format에 맞춘 Prompt 형식으로 구성합니다.
def format_qa_prompt(row):
    return {
        "text": f"<|begin_of_text|>"
                f"<|start_header_id|>system<|end_header_id|>주어진 정보를 바탕으로 질문에 답하세요.<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>정보:\n{row['context']}\n질문:\n{row['question']}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>{row['answer']}<|eot_id|>",
        "task_type": "qa"
    }

# 데이터 포맷팅 후 최대 길이보다 긴 문장 제거
def filter_long_texts(df, max_length):
    return df[df['text'].str.len() <= max_length]

# QA 데이터 포맷팅
df_qa_formatted = df_qa.apply(format_qa_prompt, axis=1, result_type='expand')
# 최대 길이보다 긴 문장 제거
df_qa_formatted = filter_long_texts(df_qa_formatted, MAX_LENGTH)
# 데이터 일부만 샘플링
df_qa_formatted = df_qa_formatted.sample(frac=0.5, random_state=42).reset_index(drop=True)
df_qa_formatted.head(2)

,text,task_type
0,<|begin_of_text|><|start_header_id|>system<|en...,qa
1,<|begin_of_text|><|start_header_id|>system<|en...,qa


In [11]:
# 첫번째 데이터 전체 확인
print(df_qa_formatted['text'][0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>주어진 정보를 바탕으로 질문에 답하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>정보:
검찰조사를 받기 위해 검찰청을 찾았을 때 검찰청사 포토라인에서 한 기자의 질문에 그 기자를 노려보는 것으로 응수해 여론의 뭇매를 맞았다. 최순실게이트 핵심인물로 청문회 증인으로 채택됐으나 출석하지 않는 것은 물론 자택까지 비워 법망을 피하자 제17대 국회의원 정봉주는 현상금 펀딩 계좌를 개설했다. 이에 정봉주를 위시한 시민들로부터 현상수배되었고 2016년 12월 13일 우병우는 연합뉴스를 통해 12월 19일 열리는 청문회에 출석하겠다는 뜻을 밝혔다. 2016년 12월 13일 현재 현상금 1800만 원까지 모였고 정봉주는 우병우의 청문회 출석 여부를 확인한 뒤 모인 돈을 '박근혜정권퇴진비상국민행동'에 기부할 것이라고 밝혔다. 5차 국회 청문회가 예정 날짜보다 3일 늦춰진 2016년 12월 22일에 열렸고 이날 출석했다.
질문:
1. 2016년 12월 13일 우병우씨가 언제 열리는 청문회에 참석할 뜻을 밝혔는가?
2. 우병우의 5차 국회 청문회 날짜는 언제인가?
3. 5차 국회 청문회가 개최된 날짜는?
4. 우병우는 어느 매체를 통해 청문회 참석 여부를 전했는가?
5. 정봉주는 몇대 국회의원인가?
6. 2016년 우병우는 어느 언론사를 통해 청문회에 출석할 것을 알렸나?
7. 2016년 우병우가 출석한 5차 국회 청문회의 개최일은?
<|eot_id|><|start_header_id|>assistant<|end_header_id|>1. 12월 19일
2. 2016년 12월 22일
3. 2016년 12월 22일
4. 연합뉴스
5. 제17대
6. 연합뉴스
7. 12월 22일<|eot_id|>


#### 요약 데이터셋 포맷팅

In [12]:
def format_summary_prompt(row):
    return {
        "text": f"<|begin_of_text|>"
                f"<|start_header_id|>system<|end_header_id|>다음 정보를 적절히 요약하세요.<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>정보:\n{row['context']}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>{row['summary']}<|eot_id|>",
        "task_type": "summary"
    }

# 요약 데이터 포맷팅
df_summary_formatted = df_summary.apply(format_summary_prompt, axis=1, result_type='expand')
# 최대 길이보다 긴 문장 제거
df_summary_formatted = filter_long_texts(df_summary_formatted, MAX_LENGTH)
# 데이터 일부만 샘플링
df_summary_formatted = df_summary_formatted.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_summary_formatted.head(2)

,text,task_type
0,<|begin_of_text|><|start_header_id|>system<|en...,summary
1,<|begin_of_text|><|start_header_id|>system<|en...,summary


In [13]:
# 첫번째 데이터 전체 확인
print(df_summary_formatted['text'][0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>다음 정보를 적절히 요약하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>정보:
아산사회복지재단 아산의학상 수상자로 김빛내리(49·기초과학연구원 RNA연구단장) 서울대 생명과학부 석좌교수와 김종성(63) 울산대 의대 신경과학교실 교수가 선정됐다. 아산재단은 21일 서울 용산구 한남동 그랜드하얏트호텔에서 제12회 아산의학상 시상식을 열고 김빛내리 석좌교수와 김종성 교수를 각각 기초의학과 임상의학 부문 수상자로 발표했다. 아산의학상은 기초의학과 임상의학 분야에서 뛰어난 업적을 이뤄낸 국내외 의과학자를 발굴해 격려하기 위한 상으로 지난 2007년 제정됐으며 상금은 부문별로 각각 3억원이다. 김 석좌교수는 세포 안에서 유전자를 조절하는 '마이크로 RNA'가 만들어지고 작동하는 원리를 규명했다. RNA의 분해를 제어해 유전자 조절에 중요한 역할을 하는 'RNA 혼합꼬리'를 발견한 공로를 인정받았다. 김 교수는 뇌졸중 후 발생하는 감정조절 장애를 체계화하고 효과적인 치료법을 개발한 업적을 높이 평가받았다. 만 40세 이하의 의과학자에게 주어지는 젊은의학자 부문 수상자로는 한범(40) 서울대 의대 의과학과 교수와 이은지(39) 분당서울대병원 안과 교수가 선정됐다. 이들에게는 상금 5,000만원이 주어진다. 한 교수는 유전체 빅데이터 분석으로 질병의 세부 특성을 구분하는 의학통계 알고리즘을 개발했다. 이 교수는 녹내장 발병 원인을 재규명하고 새로운 진단과 치료의 근거를 마련했다는 평가를 받았다.<|eot_id|><|start_header_id|>assistant<|end_header_id|>김빛내리 석좌교수와 김종성 교수는 각각 '마이크로 RNA'의 생성과 작동원리를 밝히고 뇌졸중 후의 감정조절 장애의 치료법을  만든 업적을 바탕으로 21일 서울 용산구 한남동 그랜드하얏트호텔에서 제 12회 아산의학상 수상자로 선정되었다.<|

### 데이터셋 통합 및 저장

In [14]:
# 데이터셋 통합
df_instruction = pd.concat([df_qa_formatted, df_summary_formatted], ignore_index=True)

# 통합된 데이터셋 저장
df_instruction.to_csv(os.path.join(PROCESSED_DATA_PATH, 'kr_finetuning_dataset.csv'), index=False, encoding='utf-8')

print(f"통합 데이터셋 크기: {df_instruction.shape}")

통합 데이터셋 크기: (4835, 2)


### 5. Chosen/Rejected 데이터셋 구성 (DPO용)

DPO (Direct Preference Optimization)를 위한 선호(chosen), 비선호(rejected) 데이터셋을 구성합니다.

- 기존 데이터셋(Human-Like-DPO-Dataset)을 활용하거나,
- 직접 instruct 모델을 사용해 생성하여 구성 가능합니다.

여기서는 기존 Human-Like 데이터셋을 활용하는 방식을 제공합니다.

In [15]:
# 기존 선호/비선호 데이터셋 로드
# https://huggingface.co/datasets/mncai/orca_dpo_pairs_ko
df_preference = pd.read_csv(os.path.join(RAW_DATA_PATH, 'dpo_raw.csv'))
print(df_preference.head(2))


                                              prompt  \
0  당신은 AI 비서입니다. 당신에게 과제가 주어질 것입니다. 상세하고 긴 답변을 생성...   
1  당신은 항상 설명을 해주는 도움이 되는 조력자입니다. 다섯 살짜리 아이에게 대답하는...   

                                              chosen  \
0        한여름집은 올바원 근처에 위치한 고객 평점 3/5의 적당한 가격의 중식당이다.   
1  C. 그런 다음 그녀는 바늘을 잉크에 담그고 연필로 다리에 디자인을 그려 마지막에는...   

                                            rejected  
0  물론이죠! 여기 당신이 제공한 모든 데이터를 설명하는 문장이 있습니다:\n\n한여름...  
1  생각 좀 해볼게요! 좋아요, 다음에 무슨 일이 일어날지 알아요! *신나게* 이야기에...  


### DPO용 데이터셋 포맷팅

In [16]:
def format_dpo_prompt(row):
    return {
        "prompt": "<|begin_of_text|>"
                  "<|start_header_id|>system<|end_header_id|>You are the user's friend.<|eot_id|>"
                  f"<|start_header_id|>user<|end_header_id|>{row['prompt']}<|eot_id|>"
                  "<|start_header_id|>assistant<|end_header_id|>",
        "chosen": row['chosen'],
        "rejected": row['rejected']
    }

def filter_dpo_texts(df, max_length):
    return df[df['prompt'].str.len() <= max_length]

# DPO 데이터 포맷팅 및 최대 길이 필터링
df_preference_formatted = df_preference.apply(format_dpo_prompt, axis=1, result_type='expand')
df_preference_formatted = filter_dpo_texts(df_preference_formatted, MAX_LENGTH)

# 데이터셋 저장
df_preference_formatted.to_csv(os.path.join(PROCESSED_DATA_PATH, 'preference_dataset.csv'), index=False, encoding='utf-8')

print(f"DPO 데이터셋 크기: {df_preference_formatted.shape}")

DPO 데이터셋 크기: (8764, 3)


In [17]:
df_preference_formatted

,prompt,chosen,rejected
0,<|begin_of_text|><|start_header_id|>system<|en...,한여름집은 올바원 근처에 위치한 고객 평점 3/5의 적당한 가격의 중식당이다.,물론이죠! 여기 당신이 제공한 모든 데이터를 설명하는 문장이 있습니다:\n\n한여름...
1,<|begin_of_text|><|start_header_id|>system<|en...,C. 그런 다음 그녀는 바늘을 잉크에 담그고 연필로 다리에 디자인을 그려 마지막에는...,"생각 좀 해볼게요! 좋아요, 다음에 무슨 일이 일어날지 알아요! *신나게* 이야기에..."
2,<|begin_of_text|><|start_header_id|>system<|en...,"이 글을 바탕으로 1901년 호주 연방의 주요 동기와 결과, 연방 정부의 역할과 책...",그렇고말고! 다음은 귀하가 제공한 질문에 대한 상세하고 긴 답변입니다:\n\n질문:...
3,<|begin_of_text|><|start_header_id|>system<|en...,"제임스는 부캐릭터들에게 매회 15,000달러를 지불한다. 단역 캐릭터가 4명이기 때...","물론이죠, 기꺼이 도와드리겠습니다! 제임스가 회당 얼마를 받는지 계산하려면, 우리는..."
4,<|begin_of_text|><|start_header_id|>system<|en...,질문:. 여자 비치발리볼 선수가 경기에서 효과적으로 공을 서브할 수 있는 적절한 기...,"물론이죠, 기꺼이 도와드리겠습니다! 의식의 흐름에 근거한 합리적인 문답은 다음과 같..."
...,...,...,...
10551,<|begin_of_text|><|start_header_id|>system<|en...,싱가포르를 이끌고 있는 토니 탄 자바 출신의 반유마산족은 주로 영어를 구사하며 자바...,"네, 여기에 제공된 모든 데이터를 설명하는 문장이 있습니다:\n\n싱가포르에서는 영..."
10552,<|begin_of_text|><|start_header_id|>system<|en...,"본명이 미셸 데미트리 찰후브인 오마르 샤리프는 '아라비아의 로렌스', '닥터 지바고...","아, 매력적인 질문이네요! 유명한 배우 오마르 샤리프는 이집트 출신이고, 이집트의 ..."
10553,<|begin_of_text|><|start_header_id|>system<|en...,1단계: 번역이 쉽도록 문장을 작은 부분으로 분해합니다:\n- - În consec...,"물론이죠, 기꺼이 도와드리겠습니다! 당신이 제공한 문장의 번역은 다음과 같습니다:\..."
10554,<|begin_of_text|><|start_header_id|>system<|en...,물론입니다,"제공된 리뷰를 바탕으로 ""네""라는 긍정적인 답변이 강한 친구에게 이 앱을 추천하고 ..."


### 최종 데이터셋 확인

In [18]:
# 저장된 데이터셋 로딩 후 확인
df_instruction_loaded = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, 'kr_finetuning_dataset.csv'))
df_preference_loaded = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, 'preference_dataset.csv'))

print("Instruction 데이터셋 예시:")
print(df_instruction_loaded.head(2))

print("\nDPO 데이터셋 예시:")
print(df_preference_loaded.head(2))

Instruction 데이터셋 예시:
                                                text task_type
0  <|begin_of_text|><|start_header_id|>system<|en...        qa
1  <|begin_of_text|><|start_header_id|>system<|en...        qa

DPO 데이터셋 예시:
                                              prompt  \
0  <|begin_of_text|><|start_header_id|>system<|en...   
1  <|begin_of_text|><|start_header_id|>system<|en...   

                                              chosen  \
0        한여름집은 올바원 근처에 위치한 고객 평점 3/5의 적당한 가격의 중식당이다.   
1  C. 그런 다음 그녀는 바늘을 잉크에 담그고 연필로 다리에 디자인을 그려 마지막에는...   

                                            rejected  
0  물론이죠! 여기 당신이 제공한 모든 데이터를 설명하는 문장이 있습니다:\n\n한여름...  
1  생각 좀 해볼게요! 좋아요, 다음에 무슨 일이 일어날지 알아요! *신나게* 이야기에...  
